In [5]:
##9.4

In [1]:
import numpy as np

R = np.array([
    [4, np.nan, np.nan, 2, np.nan],
    [np.nan, 5, np.nan, 3, 1],
    [np.nan, np.nan, 3, 4, 4],
    [5, 2, 1, 2, np.nan]
])

num_users, num_items = R.shape
K = 3

np.random.seed(1)
P = np.random.normal(scale=1.0 / K, size=(num_users, K))
Q = np.random.normal(scale=1.0 / K, size=(num_items, K))

In [2]:
from sklearn.metrics import mean_squared_error

def get_rmse(R, P, Q, non_zeros):
    full_pred_matrix = np.dot(P, Q.T)

    x_non_zero_ind = [nz[0] for nz in non_zeros]
    y_non_zero_ind = [nz[1] for nz in non_zeros]

    R_non_zeros = R[x_non_zero_ind, y_non_zero_ind]
    full_pred_matrix_non_zeros = full_pred_matrix[x_non_zero_ind, y_non_zero_ind]

    mse = mean_squared_error(R_non_zeros, full_pred_matrix_non_zeros)
    rmse = np.sqrt(mse)
    return rmse

In [3]:
non_zeros = [
    (i, j, R[i, j])
    for i in range(num_users)
    for j in range(num_items)
    if R[i, j] > 0
]

steps = 1000
learning_rate = 0.01
r_lambda = 0.01

for step in range(steps):
    for i, j, r in non_zeros:
        eij = r - np.dot(P[i, :], Q[j, :].T)

        P[i, :] = P[i, :] + learning_rate * (eij * Q[j, :] - r_lambda * P[i, :])
        Q[j, :] = Q[j, :] + learning_rate * (eij * P[i, :] - r_lambda * Q[j, :])

    rmse = get_rmse(R, P, Q, non_zeros)

    if step % 50 == 0:
        print("### iteration step:", step, "rmse:", rmse)

### iteration step: 0 rmse: 3.2388050277987723
### iteration step: 50 rmse: 0.4876723101369648
### iteration step: 100 rmse: 0.1564340384819247
### iteration step: 150 rmse: 0.07455141311978046
### iteration step: 200 rmse: 0.04325226798579314
### iteration step: 250 rmse: 0.029248328780878973
### iteration step: 300 rmse: 0.022621116143829466
### iteration step: 350 rmse: 0.019493636196525135
### iteration step: 400 rmse: 0.018022719092132704
### iteration step: 450 rmse: 0.01731968595344266
### iteration step: 500 rmse: 0.016973657887570753
### iteration step: 550 rmse: 0.016796804595895633
### iteration step: 600 rmse: 0.01670132290188466
### iteration step: 650 rmse: 0.01664473691247669
### iteration step: 700 rmse: 0.016605910068210026
### iteration step: 750 rmse: 0.016574200475705
### iteration step: 800 rmse: 0.01654431582921597
### iteration step: 850 rmse: 0.01651375177473524
### iteration step: 900 rmse: 0.01648146573819501
### iteration step: 950 rmse: 0.016447171683479155


In [4]:
pred_matrix = np.dot(P, Q.T)
print('예측 행렬:\n', np.round(pred_matrix, 3))

예측 행렬:
 [[3.991 0.897 1.306 2.002 1.663]
 [6.696 4.978 0.979 2.981 1.003]
 [6.677 0.391 2.987 3.977 3.986]
 [4.968 2.005 1.006 2.017 1.14 ]]


In [6]:
##9.8

In [1]:
from surprise import SVD
from surprise import Dataset 
from surprise import accuracy 
from surprise.model_selection import train_test_split

In [2]:
from surprise import Dataset
from surprise.model_selection import train_test_split

data = Dataset.load_builtin('ml-100k')
trainset, testset = train_test_split(data, test_size=0.25, random_state=0)

Dataset ml-100k could not be found. Do you want to download it? [Y/n] 

 Y


Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to C:\Users\minse/.surprise_data/ml-100k


In [3]:
algo = SVD(random_state=0)
algo.fit(trainset)

In [4]:
predictions = algo.test(testset)
accuracy.rmse(predictions)

RMSE: 0.9467


0.9466860806937948

In [5]:
[(pred.uid, pred.iid, pred.est) for pred in predictions[:3]]

[('120', '282', 3.5114147666251547),
 ('882', '291', 3.573872419581491),
 ('535', '507', 4.033583485472447)]

In [6]:
uid = str(196)
iid = str(302)

pred = algo.predict(uid, iid)
print(pred)

user: 196        item: 302        r_ui = None   est = 4.49   {'was_impossible': False}


In [7]:
accuracy.rmse(predictions)

RMSE: 0.9467


0.9466860806937948

In [2]:
import pandas as pd

ratings = pd.read_csv('ratings.csv')
ratings.to_csv('ratings_noh.csv', index=False, header=False)

In [4]:
from surprise import Reader, Dataset

reader = Reader(
    line_format='user item rating timestamp',
    sep=',',
    rating_scale=(0.5, 5)
)

data = Dataset.load_from_file(
    'ratings_noh.csv',
    reader=reader
)

In [5]:
from surprise import SVD, accuracy
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(data, test_size=0.25, random_state=0)

algo = SVD(n_factors=50, random_state=0)
algo.fit(trainset)

predictions = algo.test(testset)
accuracy.rmse(predictions)

RMSE: 0.8682


0.8681952927143516

In [7]:
import pandas as pd
from surprise import Reader, Dataset, SVD, accuracy
from surprise.model_selection import train_test_split

ratings = pd.read_csv('ratings.csv')

reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

trainset, testset = train_test_split(data, test_size=0.25, random_state=0)

algo = SVD(n_factors=50, random_state=0)
algo.fit(trainset)

predictions = algo.test(testset)
accuracy.rmse(predictions)

RMSE: 0.8682


0.8681952927143516

In [8]:
import pandas as pd
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate

ratings = pd.read_csv('ratings.csv')

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

algo = SVD(random_state=0)

cross_validate(
    algo,
    data,
    measures=['RMSE', 'MAE'],
    cv=5,
    verbose=True
)


Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8779  0.8727  0.8697  0.8745  0.8737  0.8737  0.0027  
MAE (testset)     0.6745  0.6707  0.6674  0.6716  0.6726  0.6714  0.0023  
Fit time          1.10    1.09    1.21    1.07    1.09    1.11    0.05    
Test time         0.10    0.14    0.13    0.10    0.12    0.12    0.02    


{'test_rmse': array([0.87794947, 0.87271543, 0.86972866, 0.87447005, 0.87366935]),
 'test_mae': array([0.6745173 , 0.67068653, 0.66744173, 0.67155919, 0.67255334]),
 'fit_time': (1.1034810543060303,
  1.0949907302856445,
  1.2058143615722656,
  1.0680840015411377,
  1.0884008407592773),
 'test_time': (0.10010457038879395,
  0.13918471336364746,
  0.12527012825012207,
  0.09952974319458008,
  0.11676263809204102)}

In [9]:
from surprise import SVD
from surprise.model_selection import GridSearchCV

param_grid = {
    'n_epochs': [20, 40, 60],
    'n_factors': [50, 100, 200]
}

gs = GridSearchCV(
    SVD,
    param_grid,
    measures=['rmse', 'mae'],
    cv=3
)

gs.fit(data)

print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.8788411958272716
{'n_epochs': 20, 'n_factors': 50}


In [10]:
from surprise import Dataset, SVD

data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

trainset = data.build_full_trainset()

algo = SVD(n_factors=50, random_state=0)
algo.fit(trainset)

In [11]:
from surprise import Reader
from surprise.dataset import DatasetAutoFolds

reader = Reader(
    line_format='user item rating timestamp',
    sep=',',
    rating_scale=(0.5, 5)
)

data_folds = DatasetAutoFolds(
    ratings_file='ratings_noh.csv',
    reader=reader
)

trainset = data_folds.build_full_trainset()

In [12]:
algo = SVD(random_state=0)
algo.fit(trainset)

In [20]:
import pandas as pd

movies = pd.read_csv('movies.csv')

movie_ids = ratings[ratings['userId'] == 9]['movieId']

if movie_ids[movie_ids == 42].count() == 0:
    print('사용자 아이디 9는 영화 아이디 42의 평점 없음')
    print(movies[movies['id'] == 42])

사용자 아이디 9는 영화 아이디 42의 평점 없음
Empty DataFrame
Columns: [index, budget, genres, homepage, id, keywords, original_language, original_title, overview, popularity, production_companies, production_countries, release_date, revenue, runtime, spoken_languages, status, tagline, title, vote_average, vote_count, cast, crew, director]
Index: []

[0 rows x 24 columns]


In [21]:
uid = str(9) 
iid = str(42)
pred = algo.predict(uid, iid, verbose=True)

user: 9          item: 42         r_ui = None   est = 3.16   {'was_impossible': False}


In [23]:
def get_unseen_surprise(ratings, movies, userId):
    seen_movies = ratings[ratings['userId'] == userId]['movieId'].tolist()
    total_movies = movies['id'].tolist()
    unseen_movies = [movie for movie in total_movies if movie not in seen_movies]

    print(
        '평점 매긴 영화 수:', len(seen_movies),
        '추천 대상 영화 수:', len(unseen_movies),
        '전체 영화 수:', len(total_movies)
    )

    return unseen_movies

unseen_movies = get_unseen_surprise(ratings, movies, 9)

평점 매긴 영화 수: 46 추천 대상 영화 수: 4796 전체 영화 수: 4803


In [27]:
def recomm_movie_by_surprise(algo, userId, unseen_movies, top_n=10):
    predictions = [
        algo.predict(str(userId), str(movieId))
        for movieId in unseen_movies
    ]

    predictions.sort(key=lambda x: x.est, reverse=True)
    top_predictions = predictions[:top_n]

    top_movie_ids = [int(pred.iid) for pred in top_predictions]
    top_movie_ratings = [pred.est for pred in top_predictions]
    top_movie_titles = movies[movies['id'].isin(top_movie_ids)]['title'].tolist()

    top_movie_preds = list(
        zip(top_movie_ids, top_movie_titles, top_movie_ratings)
    )

    return top_movie_preds


unseen_movies = get_unseen_surprise(ratings, movies, 9)
top_movie_preds = recomm_movie_by_surprise(algo, 9, unseen_movies, top_n=10)

print('##### Top-10 추천 영화 리스트 #####')
for movie in top_movie_preds:
    print(movie[1], movie[2])

평점 매긴 영화 수: 46 추천 대상 영화 수: 4796 전체 영화 수: 4803
##### Top-10 추천 영화 리스트 #####
Pirates of the Caribbean: Dead Man's Chest 4.2734946556067275
Terminator 3: Rise of the Machines 4.036503196978803
Beverly Hills Cop III 4.024138153352864
My Best Friend's Wedding 3.990854649798709
The Talented Mr. Ripley 3.939585319618032
License to Wed 3.9065216752530367
Bridesmaids 3.8835049402413895
The Good Thief 3.8418724049818347
We're No Angels 3.838426660560348
The Remains of the Day 3.8213018092260733
